In [3]:
from collections import defaultdict
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from torch import Tensor
from torch_geometric.nn import DistMult
from torch.utils.data import TensorDataset, DataLoader
from tqdm import tqdm
from sklearn.manifold import TSNE

device = 'cuda' if torch.cuda.is_available() else 'cpu'



In [19]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)

df = pd.read_csv('../data/edges/clean_triples.csv')

node_keys, node_values = pd.factorize(pd.concat([df['id_entity_1'], df['id_entity_2']]))
rawid2id = {k: v.item() for v, k in zip(node_keys, node_values)}

pred_keys, pred_values = pd.factorize(df['predicate'])
pred2id = {k: v.item() for v, k in zip(pred_keys, pred_values)}

df['id_entity_1'] = df['id_entity_1'].apply(lambda x: rawid2id[x])
df['id_entity_2'] = df['id_entity_2'].apply(lambda x: rawid2id[x])
df['predicate'] = df['predicate'].apply(lambda x: pred2id[x])

hrt_arr = np.array([df['id_entity_1'].to_numpy(), df['predicate'].to_numpy(), df['id_entity_2'].to_numpy()])
hrt_tensor = torch.tensor(hrt_arr, dtype=torch.long).t()
hrt_tensor_inv = torch.flip(hrt_tensor, dims=[0])
hrt_tensor = torch.vstack([hrt_tensor, hrt_tensor_inv])


num_triples = hrt_tensor.shape[0]

indices = torch.randperm(num_triples)
train_size = int(0.8 * num_triples)
val_size = int(0.1 * num_triples)

test_indices = indices[train_size + val_size:]
val_indices = indices[train_size : train_size + val_size]
train_indices = indices[:train_size]

train_triplets = hrt_tensor[train_indices].to(device)
val_triplets = hrt_tensor[val_indices].to(device)
test_triplets = hrt_tensor[test_indices].to(device)


filtered_dict = defaultdict(set)

# Обязательно переводим тензор на CPU и конвертируем в список Python
triplets_list = hrt_tensor.cpu().tolist()

for h, r, t in triplets_list:
    filtered_dict[(h, r)].add(t)

In [25]:
class CustomDistMult(DistMult):
    def test(self, head_index, rel_type, tail_index, batch_size=128, filtered_dict = None, k_list=[1, 5, 10, 50]):
        self.eval()


        num_queries = head_index.numel()
        ranks = torch.zeros(num_queries, device=head_index.device)

        #получаем эмбеддинги ВСЕХ узлов графа один раз
        #E_all размерности [num_nodes, embedding_dim]
        with torch.no_grad():
            E_all = self.node_emb.weight.data

            #разбиваем сами запросы (h, r) на батчи, а не хвосты
            for i, start_idx in enumerate(tqdm(range(0, num_queries, batch_size))):
                if i >=50:
                    break
                end_idx = min(start_idx + batch_size, num_queries)

                h_batch = head_index[start_idx:end_idx]
                r_batch = rel_type[start_idx:end_idx]
                t_batch = tail_index[start_idx:end_idx] #целевые хвосты

                #получаем эмбеддинги для текущего батча запросов
                h_emb = self.node_emb.weight.data[h_batch]
                r_emb = self.rel_emb.weight.data[r_batch]

                #query_emb: [current_batch_size, embedding_dim]
                query_emb = h_emb * r_emb

                #умножаем батч запросов на транспонированную матрицу всех узлов
                #scores: [current_batch_size, num_nodes]
                scores = torch.matmul(query_emb, E_all.T)

                #фильтрация и маскирование для батча
                if filtered_dict is not None:
                    mask_batch = []
                    mask_idx = []
                    for i_in_batch, (h, r, t) in enumerate(zip(h_batch, r_batch, t_batch)):
                        h, r, t = int(h), int(r), int(t)
                        true_tails = filtered_dict.get((h, r), [])

                        for true_t in true_tails:
                            if true_t != t:
                                mask_batch.append(i_in_batch)
                                mask_idx.append(true_t)

                    if mask_batch:
                        scores[mask_batch, mask_idx] = -float('inf')

                #выделяем целевые скоры [current_batch_size, 1]
                target_scores = scores[torch.arange(len(t_batch)), t_batch].unsqueeze(1)

                #cчитаем ранги для всего батча
                batch_ranks = (scores > target_scores).sum(dim=1) + 1
                ranks[start_idx:end_idx] = batch_ranks

        #итоговый подсчет метрик
        mrr = (1.0 / ranks).float().mean().item()

        hits_at_k = {}
        for k in k_list:
            hits_at_k[k] = (ranks <= k).float().mean().item()

        formatted_hits = {k: f"{v:.4f}" for k, v in hits_at_k.items()}

        return {'MRR': mrr, 'Hits': formatted_hits}

    def loss(
        self,
        head_index: Tensor,
        rel_type: Tensor,
        tail_index: Tensor,
    ) -> Tensor:

        pos_score = self(head_index, rel_type, tail_index)
        neg_score = self(*self.sns_sample(head_index, rel_type, tail_index))

        return F.margin_ranking_loss(
            pos_score,
            neg_score,
            target=torch.ones_like(pos_score),
            margin=self.margin,
        )


    def get_sns_negatives(self, all_embs, pos_indices, n1, n2):
        """Вспомогательная функция для поиска сложных негативов"""
        # Сэмплируем N1 кандидатов для каждого триплета в батче сразу
        # shape: (num_negatives, n1)
        cand_indices = torch.randint(0, self.num_nodes, (pos_indices.size(0), n1), device=self.node_emb.weight.device)

        # Получаем эмбеддинги: позитивных сущностей и кандидатов
        pos_embs = all_embs[pos_indices].unsqueeze(1)    # (num_negatives, 1, dim)
        cand_embs = all_embs[cand_indices]                # (num_negatives, n1, dim)

        # Считаем расстояние d = ||pos - cand||
        dist = torch.norm(pos_embs - cand_embs, p=2, dim=-1) # (num_negatives, n1)

        # Считаем вероятности P = softmax(1/d)
        probs = torch.softmax(1.0 / (dist + 1e-9), dim=1)

        # Выбираем N2 лучших (самых близких) из N1
        _, top_n2_loc_idx = torch.topk(probs, k=n2, dim=1)

        # Из N2 выбираем по 1 случайному индексу для каждого примера (Exploration)
        rand_selector = torch.randint(0, n2, (pos_indices.size(0),), device=self.node_emb.weight.device)

        # Собираем финальные индексы
        final_loc_idx = top_n2_loc_idx[torch.arange(pos_indices.size(0)), rand_selector]
        return cand_indices[torch.arange(pos_indices.size(0)), final_loc_idx]



    @torch.no_grad()
    def sns_sample(
        self,
        head_index: torch.Tensor,
        rel_type: torch.Tensor,
        tail_index: torch.Tensor,
        n1: int = 50,  # Размер начального пула кандидатов
        n2: int = 5   # Размер пула "сложных" негативов
        ):
        # 1. Получаем текущие эмбеддинги всех сущностей
        # Предполагаем, что они лежат в self.node_emb.weight
        all_embs = self.node_emb.weight
        batch_size = head_index.numel()
        num_negatives = batch_size // 2

        # Клонируем индексы для модификации
        new_head = head_index.clone()
        new_tail = tail_index.clone()


        # 2. Коррептируем головы (первая половина батча)
        new_head[:num_negatives] = self.get_sns_negatives(all_embs, head_index[:num_negatives], n1, n2)

        # 3. Коррептируем хвосты (вторая половина батча)
        new_tail[num_negatives:] = self.get_sns_negatives(all_embs, tail_index[num_negatives:], n1, n2)

        return new_head, rel_type, new_tail


In [26]:
model = CustomDistMult(
    num_nodes=len(rawid2id),
    num_relations = len(pred2id),
    hidden_channels=128,
    margin=1
).to(device)

In [27]:
dataset = TensorDataset(train_triplets)
dataloader = DataLoader(dataset, batch_size=8192, shuffle=True)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

model.train()
for epoch in tqdm(range(3)):
    for batch in dataloader:
        h_train = batch[0][:, 0]
        r_train = batch[0][:, 1]
        t_train = batch[0][:, 2]

        optimizer.zero_grad()
        loss = model.loss(h_train, r_train, t_train)
        loss.backward()
        optimizer.step()
    print(loss.item())


#h_test = test_triplets[:, 0]
#r_test = test_triplets[:, 1]
#t_test = test_triplets[:, 2]


#model.eval()
#with torch.no_grad():
#    print(model.test(h_test, r_test, t_test, 8192, filtered_dict=None, sampling=False))

 33%|███▎      | 1/3 [00:51<01:43, 51.90s/it]

0.026800861582159996


 67%|██████▋   | 2/3 [01:44<00:52, 52.11s/it]

0.011729404330253601


100%|██████████| 3/3 [02:36<00:00, 52.06s/it]

0.009974426589906216


In [28]:
h_test = test_triplets[:, 0]
r_test = test_triplets[:, 1]
t_test = test_triplets[:, 2]


model.eval()
with torch.no_grad():
   print(model.test(h_test, r_test, t_test, batch_size=64, filtered_dict=filtered_dict))

  0%|          | 50/15827 [00:03<16:26, 15.99it/s] 

{'MRR': inf, 'Hits': {1: '0.9978', 5: '0.9981', 10: '0.9982', 50: '0.9986'}}


In [11]:
X_emb = model.node_emb.weight
X_emb = X_emb.detach().cpu().numpy()

In [ ]:
tsne = TSNE(n_components=2)

X_embedded = tsne.fit_transform(X_emb)